# Project 13 — Local Multi-PDF Research Librarian

## Answer Questions Across Multiple Papers with Evidence

**Goal:** Build a local research assistant that indexes multiple academic papers,
answers cross-paper questions with evidence citations, and synthesizes findings
across documents — all running locally with LlamaIndex and Ollama.

**Stack:** Ollama · LlamaIndex · Jupyter

```
Paper 1 ─┐
Paper 2 ─┼──► LlamaIndex Documents ──► VectorStoreIndex ──► Ollama Embeddings
Paper 3 ─┘                                                        │
                                                             stored vectors
                                                                   │
Research question ──► query engine ──► top nodes ──► LLM ──► Cited Answer
```

### What You'll Learn

1. Create multi-document LlamaIndex indices with paper metadata
2. Query across papers with source attribution
3. Use CitationQueryEngine for inline references
4. Synthesize findings from multiple sources
5. Compare findings across papers

### Prerequisites

- **Ollama** running with `nomic-embed-text` and `qwen3:8b`
- Python 3.9+

In [ ]:
# Install dependencies (uncomment and run once)
!pip install -q llama-index llama-index-llms-ollama llama-index-embeddings-ollama

## Step 1 — Verify Ollama Is Running

In [ ]:
import requests

OLLAMA_BASE = "http://localhost:11434"

try:
    r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    r.raise_for_status()
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"✅ Ollama is running — {len(models)} model(s) available")
    for m in models:
        print(f"   • {m}")
except Exception as e:
    print(f"❌ Cannot reach Ollama at {OLLAMA_BASE}: {e}")

## Step 2 — Configure LlamaIndex with Ollama

LlamaIndex uses `Settings` to configure the LLM and embedding model globally.
We point both at our local Ollama instance.

In [ ]:
from llama_index.core import Settings, VectorStoreIndex, Document
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

Settings.llm = Ollama(model="qwen3:8b", request_timeout=120.0)
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")

print("✅ LlamaIndex configured with local Ollama models")

## Step 3 — Create Sample Research Papers

We create 4 simulated research papers with realistic abstracts and key findings.
Each paper has metadata (paper_id, year, topic) that enables cross-referencing.

In a real project, you would load PDFs with `SimpleDirectoryReader`.

In [ ]:
papers = [
    Document(
        text=(
            "Title: Attention Mechanisms in Neural Networks\n"
            "Author: Smith et al., 2023\n\n"
            "Abstract: This paper surveys attention mechanisms from basic additive attention "
            "to multi-head self-attention used in transformers. We find that attention improves "
            "model performance by 15-30% on sequence tasks while adding minimal parameters.\n\n"
            "Key Findings:\n"
            "1. Self-attention scales quadratically with sequence length O(n²)\n"
            "2. Multi-head attention captures different types of relationships\n"
            "3. Relative position encodings outperform absolute ones by 5-8%\n"
            "4. Flash attention reduces memory from O(n²) to O(n)\n"
            "5. Sparse attention patterns can reduce compute by 40-60% with minimal quality loss"
        ),
        metadata={"paper_id": "P001", "year": 2023, "topic": "attention", "authors": "Smith et al."},
    ),
    Document(
        text=(
            "Title: Efficient Retrieval-Augmented Generation\n"
            "Author: Johnson et al., 2024\n\n"
            "Abstract: We propose improvements to RAG pipelines that reduce latency by 40% "
            "while maintaining answer quality. Key innovations include hierarchical retrieval, "
            "context compression, and adaptive chunk sizing.\n\n"
            "Key Findings:\n"
            "1. Hierarchical retrieval (summary then detail) reduces retrievals by 60%\n"
            "2. Context compression removes 40% of tokens with <2% quality loss\n"
            "3. Adaptive chunks (200-800 tokens) outperform fixed-size chunks by 15%\n"
            "4. Hybrid BM25+dense retrieval improves recall by 25% over dense alone\n"
            "5. Reranking adds 50ms latency but improves precision by 20%"
        ),
        metadata={"paper_id": "P002", "year": 2024, "topic": "RAG", "authors": "Johnson et al."},
    ),
    Document(
        text=(
            "Title: Local Language Models for Enterprise\n"
            "Author: Williams et al., 2024\n\n"
            "Abstract: We evaluate running 7B-13B parameter models locally for enterprise "
            "use cases. Results show local models achieve 85-92% of GPT-4 quality on "
            "domain-specific tasks when combined with RAG and fine-tuning.\n\n"
            "Key Findings:\n"
            "1. 7B models with RAG match GPT-4 on closed-domain QA tasks\n"
            "2. Fine-tuning on 1000 domain examples improves accuracy by 20%\n"
            "3. Quantized models (4-bit) run at 30+ tokens/sec on consumer GPUs\n"
            "4. Privacy-sensitive industries prefer local deployment 3:1 over cloud\n"
            "5. Total cost of ownership is 60% lower for local vs cloud over 3 years"
        ),
        metadata={"paper_id": "P003", "year": 2024, "topic": "local_llm", "authors": "Williams et al."},
    ),
    Document(
        text=(
            "Title: Embedding Models for Semantic Search\n"
            "Author: Chen et al., 2024\n\n"
            "Abstract: We benchmark 15 embedding models on domain-specific retrieval tasks. "
            "We find that instruction-tuned embeddings outperform general-purpose ones by "
            "12-18% on specialized corpora.\n\n"
            "Key Findings:\n"
            "1. Instruction-tuned embeddings (e.g., nomic-embed) excel on domain tasks\n"
            "2. Embedding dimension 768 provides best quality/speed tradeoff\n"
            "3. Matryoshka embeddings allow dimension reduction with <5% quality loss\n"
            "4. Domain-specific fine-tuning of embeddings improves recall by 25%\n"
            "5. Hybrid sparse+dense retrieval consistently outperforms either alone"
        ),
        metadata={"paper_id": "P004", "year": 2024, "topic": "embeddings", "authors": "Chen et al."},
    ),
]

print(f"✅ Created {len(papers)} sample research papers:")
for p in papers:
    print(f"   [{p.metadata['paper_id']}] {p.metadata['authors']} ({p.metadata['year']}) — {p.metadata['topic']}")

## Step 4 — Build Cross-Paper Index

We create a single `VectorStoreIndex` containing all papers. LlamaIndex
automatically chunks, embeds, and indexes the documents. Each chunk retains
its parent paper's metadata for attribution.

In [ ]:
index = VectorStoreIndex.from_documents(papers, show_progress=True)
query_engine = index.as_query_engine(similarity_top_k=3)
print("✅ Cross-paper index built!")

## Step 5 — Test Cross-Reference Research Questions

Let's ask questions that require the system to find and synthesize information
from multiple papers. We show both the answer and the source nodes used.

In [ ]:
research_questions = [
    "What are the key findings about attention mechanism efficiency?",
    "How do local models compare to GPT-4 for enterprise use?",
    "What techniques reduce RAG latency?",
    "What is the best embedding dimension for retrieval?",
]

for q in research_questions:
    print(f"\n{'='*60}")
    print(f"Research Q: {q}")
    response = query_engine.query(q)
    print(f"\nAnswer: {response}")
    print(f"\nEvidence from:")
    for node in response.source_nodes:
        pid = node.metadata.get('paper_id', '?')
        authors = node.metadata.get('authors', '?')
        score = f"{node.score:.4f}" if node.score is not None else "N/A"
        print(f"  [{pid}] {authors} (score={score}): {node.text[:80]}...")

## Step 6 — Citation-Aware Query Engine

LlamaIndex's `CitationQueryEngine` automatically adds inline citation
markers ([1], [2], etc.) to the generated response and maps them back
to specific source passages. This mimics academic citation style.

In [ ]:
from llama_index.core.query_engine import CitationQueryEngine

citation_engine = CitationQueryEngine.from_args(
    index, similarity_top_k=3, citation_chunk_size=256
)

synthesis_q = "Synthesize the key findings across all papers about improving AI system efficiency."
response = citation_engine.query(synthesis_q)

print("RESEARCH SYNTHESIS:")
print(response)
print("\nCitations:")
for i, node in enumerate(response.source_nodes):
    pid = node.metadata.get('paper_id', '?')
    authors = node.metadata.get('authors', '?')
    print(f"  [{i+1}] {pid} ({authors}): {node.text[:100]}...")

## Step 7 — Compare Findings Across Papers

Let's ask a comparison question that forces the system to contrast
findings from different papers.

In [ ]:
comparison_questions = [
    "Compare the findings about retrieval improvements across different papers.",
    "What do the papers collectively say about the tradeoff between speed and quality?",
]

for q in comparison_questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    response = citation_engine.query(q)
    print(f"\nA: {response}")
    papers_cited = set(n.metadata.get('paper_id', '?') for n in response.source_nodes)
    print(f"\nPapers cited: {papers_cited}")

## Step 8 — Interactive Research Helper

In [ ]:
def research(question: str) -> str:
    """Ask the research librarian a question with citations."""
    response = citation_engine.query(question)
    print(f"Answer:\n{response}\n")
    print(f"Sources ({len(response.source_nodes)} passages):")
    for i, node in enumerate(response.source_nodes):
        pid = node.metadata.get('paper_id', '?')
        authors = node.metadata.get('authors', '?')
        print(f"   [{i+1}] {pid} ({authors}): {node.text[:80]}...")
    return str(response)

_ = research("What are the cost benefits of local LLM deployment?")

## Limitations & Tradeoffs

| Aspect | What happens | How to improve |
|--------|-------------|----------------|
| **Simulated papers** | We used text — real papers would be PDFs | Use `SimpleDirectoryReader` with PDF parsing |
| **Small corpus** | Only 4 papers — real libraries have thousands | Add hierarchical indexing or summary indices |
| **Citation granularity** | Citations point to chunks, not exact sentences | Use smaller citation_chunk_size |
| **Cross-paper synthesis** | Quality depends on retrieval coverage | Increase similarity_top_k or use reranking |
| **Conflicting findings** | Papers may disagree — LLM may not handle well | Add explicit conflict-detection prompting |

## What You Learned

1. **Multi-document LlamaIndex index** — building a cross-paper search system
2. **Paper metadata** — tracking paper_id, authors, year, topic for attribution
3. **Cross-reference queries** — answering questions that span multiple papers
4. **CitationQueryEngine** — generating inline citations with source mapping
5. **Research synthesis** — combining evidence from multiple sources

## Exercises

1. **Add real PDFs** — download a few arXiv papers and index them with `SimpleDirectoryReader`
2. **Increase corpus** — add 10 more papers and test retrieval quality
3. **Summary index** — build a `SummaryIndex` alongside the vector index
4. **Conflict detection** — prompt the LLM to flag when papers disagree
5. **Export bibliography** — generate a reference list from cited papers

---

*Next project: **14 — Local Financial Report Analyst***